In [1]:
import json
from datasets import load_dataset, DatasetDict
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm
from rag_bench import evaluator, data, helper
from sentence_transformers import SentenceTransformer

from embedder import insert_dataset

/home/batori/llm/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Загрузка датасетов

In [2]:
CACHE_DIR = "../datasets/"

HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"

PUBLIC_TEXTS_REPO: str = "ai-forever/hist-rag-bench-public-texts"
PUBLIC_QA_REPO: str = "ai-forever/hist-rag-bench-public-questions"


def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version, cache_dir=CACHE_DIR)


def get_private_texts_dataset(version):
    return load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version, cache_dir=CACHE_DIR)

# map public_id:private_id
def get_public_to_private_texts_mapping(version):
    private_texts_ds = get_private_texts_dataset(version)
    mapping = {}
    for item in private_texts_ds["train"]:
        mapping[item["public_id"]] = item["id"]
    return mapping


version = helper.get_latest_version(helper.get_ds_versions(PUBLIC_TEXTS_REPO))
texts_ds = load_dataset(PUBLIC_TEXTS_REPO, revision=version, cache_dir=CACHE_DIR)
questions_ds = load_dataset(PUBLIC_QA_REPO, revision=version, cache_dir=CACHE_DIR)
private_texts_ds = get_private_texts_dataset(version)
qa_dataset = get_private_qa_dataset(version)
mapping = get_public_to_private_texts_mapping(version)

### Создание базы

In [3]:
COLLECTION_NAME: str = "dragon"
CHROMA_PATH: str = "../chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = None

try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception as e:
    print(f"Коллекция {COLLECTION_NAME} уже существует и будет перезатерта: {e}")
finally:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine",
        }
    )
    print(f"Коллекция {COLLECTION_NAME} создана по пути {CHROMA_PATH}")

# embending model
model = SentenceTransformer('intfloat/multilingual-e5-small')
insert_dataset(dataset=texts_ds, collection=collection, model=model)

Коллекция dragon создана по пути ../chroma_db


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 589.00it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Added batch 1, size: 663
Всего добавлено 663 чанков в коллекцию dragon


### Оценка ретривера

In [4]:
from retriever import ChromaRetriever

def get_retrivier_result(query, collection):
    results = ChromaRetriever(collection).top_n(query=query, n=3)
    found_ids = [int(x["id"]) for x in results["metadatas"][0]]
    relevant_chunks = results['documents'][0]
    metadatas = results['metadatas'][0]
    return found_ids, relevant_chunks, metadatas

def create_context(q_id, relevant_chunks, metadatas, separator="\n\n---\n\n"):
    contexts_data = {}
    contexts_data[q_id] = {
        "relevant_chunks": relevant_chunks,
        "metadatas": metadatas
    }
    context_parts = []
    for i, (chunk, metadata) in enumerate(zip(relevant_chunks, metadatas)):
        score = metadata.get('score', 0)
        context_parts.append(f"[Документ {i+1} (релевантность: {score:.3f})]\n{chunk}")
    context = separator.join(context_parts)
    return f'Контекст: {context}'

results = {}
context = {}
public_questions = [{"id": str(row["public_id"]), "question": row["question"]} for row in qa_dataset["train"]]

for q in public_questions:
    q_id = q["id"]
    question = q["question"]
    found_ids, relevant_chunks, metadatas = get_retrivier_result(question, collection)
    context[q_id] = create_context(q_id, relevant_chunks, metadatas)
    results[q_id] = {
        "found_ids": found_ids,
        "model_answer": "-",
    }

ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
ev_res["average_metrics"]["retrieval"], results

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 609.23it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


({'hit_rate': np.float64(0.8333333333333334),
  'mrr': np.float64(0.7547222222222222),
  'precision': np.float64(0.31722222222222224)},
 {'110': {'found_ids': [342, 490, 26], 'model_answer': '-'},
  '419': {'found_ids': [303, 412, 259], 'model_answer': '-'},
  '565': {'found_ids': [7, 215, 119], 'model_answer': '-'},
  '77': {'found_ids': [412, 488, 100], 'model_answer': '-'},
  '181': {'found_ids': [388, 431, 332], 'model_answer': '-'},
  '284': {'found_ids': [529, 27, 529], 'model_answer': '-'},
  '10': {'found_ids': [1, 41, 378], 'model_answer': '-'},
  '469': {'found_ids': [220, 121, 293], 'model_answer': '-'},
  '78': {'found_ids': [16, 409, 18], 'model_answer': '-'},
  '349': {'found_ids': [234, 140, 313], 'model_answer': '-'},
  '55': {'found_ids': [271, 100, 100], 'model_answer': '-'},
  '118': {'found_ids': [464, 26, 342], 'model_answer': '-'},
  '109': {'found_ids': [66, 66, 234], 'model_answer': '-'},
  '588': {'found_ids': [67, 501, 314], 'model_answer': '-'},
  '369': {'fo

### Оценка генератора

In [5]:
from generator import LLMGenerator, GigaChatGenerator
from retriever import ChromaRetriever

small_qa_ds = DatasetDict({
    "train" : qa_dataset["train"].select(range(min(3, len(qa_dataset["train"]))))
})

def create_context(relevant_chunks, metadatas, separator="\n\n---\n\n"):
    context_parts = []
    for i, (chunk, metadata) in enumerate(zip(relevant_chunks, metadatas)):
        score = metadata.get('score', 0)
        context_parts.append(f"[Документ {i+1} (релевантность: {score:.3f})]\n{chunk}")
    context = separator.join(context_parts)
    return context


def get_generator_answers(query, collection):  
    found_ids, relevant_chunks, metadatas = get_retrivier_result(query, collection)
    context = create_context(relevant_chunks, metadatas)
    answer = GigaChatGenerator().generate(context=context, query=query)
    return found_ids, answer

retriever_result = results
results = {}
public_questions = [{"id": str(row["public_id"]), "question": row["question"]} for row in small_qa_ds["train"]]

for q in public_questions:
    q_id = q["id"]
    question = q["question"]
    found_ids, answer = get_generator_answers(question, collection)
    results[q_id] = {
        "found_ids": found_ids,
        "model_answer": answer,
    }

ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())
ev_res["average_metrics"]

{'retrieval': {'hit_rate': np.float64(1.0),
  'mrr': np.float64(0.8333333333333334),
  'precision': np.float64(0.3333333333333333)},
 'generation': {'rouge1': np.float64(0.5666666666666668),
  'rougeL': np.float64(0.5666666666666668),
  'substr_match': np.float64(0.3333333333333333)}}

In [6]:
with open("dragon_chroma_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Результаты сохранены: dragon_chroma_results.json")

✅ Результаты сохранены: dragon_chroma_results.json
